<a href="https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/02_day2_pandas.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 在浏览器中打开本 notebook 后，点击菜单 **代码执行程序 → 全部运行** 即可按顺序执行所有单元格。

# Day 2｜Pandas 数据处理

**本日目标**：掌握 Pandas 的核心操作——读表、筛选、改列、分组统计、合并表。这是 20 天里最常用的一件武器，学生管理系统的统计报表、博士阶段处理实验数据都靠它。

**学完你能做到**：
- 用 `read_csv` 读取数据并快速查看
- 用布尔条件筛选出想要的行
- 用 `groupby` 做分组统计（比如各班平均分）
- 用 `merge` 把两张表拼起来

---
## ⏰ 今日学习节奏（每天 4 小时）

| 环节 | 时长 | 做什么 |
|---|---|---|
| 1. 讲解 | 约 1 小时 | 老师/家长带着过一遍本 notebook 的讲解部分，把知识点讲清楚 |
| 2. 自己练 | 约 2 小时 | 独立完成下面「✏️ 今日练习清单」，在 notebook 和**云服务器**上动手操作 |
| 3. 答疑 | 约 30–60 分钟 | 把练习中卡住的问题集中提出来答疑（问题随手记在文末「📒 我的问题清单」） |

> ✍️ **要求**：每一个练习都要真正动手写出来、跑出来，不要只看答案。

---
## 一、认识 DataFrame（20 分钟）

DataFrame 就是一张"带行列名字的表格"：每一列是一种类型，每一行是一条记录。

In [ ]:
import pandas as pd

# 用字典建一张学生成绩表
df = pd.DataFrame({
    "id":      [20260001, 20260002, 20260003, 20260004, 20260005],
    "name":    ["李雷", "韩梅梅", "Jim", "Lucy", "Lily"],
    "cls":     ["一班", "二班", "一班", "二班", "一班"],
    "math":    [88, 91, 76, 59, 95],
    "english": [92, 84, 88, 71, 60],
})
df

In [ ]:
# 拿到一张表先做这 5 件事（数据探索五连）
print(df.shape)        # (行数, 列数)
print(df.dtypes)       # 每列类型
print(df.head(3))      # 前 3 行
print(df.describe())   # 数值列的统计摘要
print(df["cls"].value_counts())   # 某列的取值计数

## 二、选列、筛选行（30 分钟，重点）

**核心心法**：`df[条件]` 筛行，`df[["列1","列2"]]` 选列，`df.loc[条件, 列]` 两者一起。

In [ ]:
# 选一列（Series）和多列（DataFrame）
print(df["math"])
print(df[["name", "math"]])

In [ ]:
# 布尔筛选：最常用的操作，没有之一
passed = df[df["math"] >= 60]
print("数学及格的人：\n", passed)

# 多条件：& 与 |，每个条件必须加括号
print("\n一班且数学及格：\n", df[(df["cls"] == "一班") & (df["math"] >= 60)])
print("\n数学或英语不及格：\n", df[(df["math"] < 60) | (df["english"] < 60)])

In [ ]:
# 排序与取前几名
print(df.sort_values("math", ascending=False))        # 按数学降序
print("\n英语前 2 名：\n", df.nlargest(2, "english"))

## 三、新增列、修改列（20 分钟）

In [ ]:
# 像做算术一样新增列
df["total"] = df["math"] + df["english"]
df["avg"]   = df["total"] / 2
df["passed"] = df["math"] >= 60          # 布尔列
df

In [ ]:
# apply + lambda：写一个自定义规则
df["level"] = df["avg"].apply(lambda x: "优秀" if x >= 90 else ("良好" if x >= 80 else ("及格" if x >= 60 else "不及格")))
df[["name", "avg", "level"]]

## 四、缺失值处理（15 分钟）

真实数据经常有缺失（空值 NaN），处理套路就三个：**发现 → 删除或填充 → 验证**。

In [ ]:
import numpy as np
df2 = df.copy()
df2.loc[1, "math"] = np.nan        # 人为制造一个缺失值
print("每列缺失数量：\n", df2.isna().sum())

df_filled = df2.fillna({"math": df2["math"].mean()})   # 用均值填充
df_dropped = df2.dropna()                               # 或直接删掉含缺失的行
print("\n填充后：\n", df_filled[["name","math"]])

## 五、分组统计 groupby（30 分钟，重点）

**心法**：`df.groupby("按谁分")["算哪列"].统计函数()` —— 比如"按班级分组，算各科平均分"。

In [ ]:
# 按班级算平均分
print(df.groupby("cls")[["math", "english"]].mean())

# 按班级算人数和最高总分
print(df.groupby("cls").agg(人数=("id", "count"), 最高总分=("total", "max"), 平均总分=("total", "mean")))

## 六、表合并 merge（15 分钟）

两张表有共同列（比如学号），就能像 SQL 的 JOIN 一样拼起来。

In [ ]:
info = pd.DataFrame({
    "id": [20260001, 20260002, 20260003, 20260004, 20260005],
    "gender": ["男", "女", "男", "女", "女"],
    "age": [17, 17, 18, 17, 18],
})
merged = df.merge(info, on="id")      # 按学号内连接
merged[["id", "name", "gender", "age", "total"]]

## 七、读写文件（10 分钟）

In [ ]:
# 读 CSV：df = pd.read_csv("文件路径")   （明天的项目会大量用到）
# 写 CSV：
merged.to_csv("students_out.csv", index=False, encoding="utf-8-sig")  # utf-8-sig 防止 Excel 打开乱码
back = pd.read_csv("students_out.csv")
back.head(3)

---
## ✏️ 今日练习清单（约 2 小时，全部动手完成）

**A 组 · 表的基本功（40 分钟）**（用下面这张表完成 A1–A4）

```python
import pandas as pd
emp = pd.DataFrame({
    "name":  ["王芳", "李强", "赵敏", "陈杰", "刘洋", "周婷"],
    "dept":  ["技术", "销售", "技术", "销售", "技术", "销售"],
    "salary": [15000, 9000, 18000, 11000, 13000, 9500],
    "age":   [28, 35, 41, 26, 33, 29],
})
```

1. 查看行数列数、每列类型、前 2 行
2. 筛选出**技术部**所有人，只显示 name 和 salary 两列
3. 找出**工资大于 10000 且年龄小于 35** 的人
4. 按工资从高到低排序，取前 3 名

**B 组 · 列与分组（40 分钟）**
- **5.** 给 emp 新增一列 `tax`：工资 × 0.1；再新增一列 `after_tax`：税后工资
- **6.** 按部门分组，计算各部门的**平均工资**和**人数**（用 agg 一次算出来）
- **7.** 另建一张部门表 `dept_info`（dept, manager：技术-张伟, 销售-李娜），merge 到 emp 上

**C 组 · 缺失值 + 综合（40 分钟）**
- **8.** 把 emp 中"赵敏"的工资设为缺失（`np.nan`），统计每列缺失数；然后用**技术部其他人的平均工资**填充这个缺失值（挑战：用 groupby/transform 或直接算出来填）
- **9.** 综合：用今天学的全部知识，对 emp 完成：筛出税后工资 > 9000 的人 → 按部门分组算平均税后工资 → 把结果保存成 `result.csv`

---
把答案写在下面的空单元格里；卡壳超过 10 分钟的题记到问题清单。

In [ ]:
import pandas as pd
import numpy as np

emp = pd.DataFrame({
    "name":  ["王芳", "李强", "赵敏", "陈杰", "刘洋", "周婷"],
    "dept":  ["技术", "销售", "技术", "销售", "技术", "销售"],
    "salary": [15000, 9000, 18000, 11000, 13000, 9500],
    "age":   [28, 35, 41, 26, 33, 29],
})

# A1

In [ ]:
# A2

In [ ]:
# A3

In [ ]:
# A4

In [ ]:
# B5

In [ ]:
# B6

In [ ]:
# B7

In [ ]:
# C8

In [ ]:
# C9

---
## 📒 我的问题清单（随手记录，答疑前整理）

学习中遇到任何卡住的地方，立刻记在下面表格里，**答疑时间集中解决**：

| 日期 | 问题描述 | 状态（待问/已解决） | 答案要点 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---
## ✅ 参考答案（先自己做，再对照）

In [ ]:
import pandas as pd
import numpy as np

emp = pd.DataFrame({
    "name":  ["王芳", "李强", "赵敏", "陈杰", "刘洋", "周婷"],
    "dept":  ["技术", "销售", "技术", "销售", "技术", "销售"],
    "salary": [15000, 9000, 18000, 11000, 13000, 9500],
    "age":   [28, 35, 41, 26, 33, 29],
})

# A1
print(emp.shape); print(emp.dtypes); print(emp.head(2))

# A2
print(emp.loc[emp["dept"] == "技术", ["name", "salary"]])

# A3
print(emp[(emp["salary"] > 10000) & (emp["age"] < 35)])

# A4
print(emp.sort_values("salary", ascending=False).head(3))

In [ ]:
# B5
emp["tax"] = emp["salary"] * 0.1
emp["after_tax"] = emp["salary"] - emp["tax"]
print(emp)

# B6
print(emp.groupby("dept").agg(平均工资=("salary", "mean"), 人数=("name", "count")))

# B7
dept_info = pd.DataFrame({"dept": ["技术", "销售"], "manager": ["张伟", "李娜"]})
print(emp.merge(dept_info, on="dept")[["name", "dept", "manager", "salary"]])

In [ ]:
# C8
emp.loc[emp["name"] == "赵敏", "salary"] = np.nan
print("缺失统计:\n", emp.isna().sum())
tech_mean = emp.loc[emp["dept"] == "技术", "salary"].mean()   # mean 自动跳过 NaN
emp["salary"] = emp["salary"].fillna(tech_mean)
print("\n填充后:\n", emp[["name", "salary"]])

# C9
emp["tax"] = emp["salary"] * 0.1
emp["after_tax"] = emp["salary"] - emp["tax"]
rich = emp[emp["after_tax"] > 9000]
result = rich.groupby("dept")["after_tax"].mean().reset_index()
result.to_csv("result.csv", index=False, encoding="utf-8-sig")
print("\n结果:\n", result)

---
➡️ **明天**：Day 3 Linux 云服务器实操 —— 打开 [https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/03_day3_linux.ipynb](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/03_day3_linux.ipynb)